# 02a — Tier 1 Bake-Off: Gemma 4 E2B vs Phi-4-mini-instruct

**Runs on:** Google Colab Free (T4 15 GB) — no other setup needed.

## How to use this notebook
1. Open in Colab. Set runtime → GPU (T4).
2. **Runtime → Run all.**
3. When prompted, upload `train.jsonl` and `val.jsonl` (from `tools/yen_sei/data/refined/` on your local machine).
4. Wait ~30–45 min. The notebook will:
   - QLoRA fine-tune both candidates on a small slice (100 train / 20 val, 1 epoch each).
   - Compare JSON-schema compliance, output length, train time, and peak VRAM.
   - Print a verdict and save `winner.json` to `/content/`.
5. Note the winner. Use it as `MODEL_NAME` in `02_train_tier1.ipynb`.

**Total user effort: pick the runtime, click Run all, drop two files. Nothing to copy-paste.**


In [ ]:
# =====================================================================
# Cell 1 — Install dependencies (silent). ~2-3 min on first run.
# =====================================================================
import subprocess, sys, os, time

PKGS = [
    "unsloth",
    "peft>=0.11.0",
    "transformers>=4.45.0",
    "datasets>=2.20.0",
    "bitsandbytes>=0.43.0",
    "accelerate>=0.34.0",
    "trl>=0.10.0",
]
print("Installing dependencies (silent)…")
t0 = time.time()
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-U", *PKGS],
    check=True,
)
print(f"Done in {time.time() - t0:.1f}s.")


In [ ]:
# =====================================================================
# Cell 2 — CONFIG. Edit only if you want different models or sizes.
# =====================================================================

# Two Tier-1 candidates from PLAN.md. Change these only if you want to
# swap one out for another HF model that Unsloth supports on a T4.
CANDIDATES = {
    "gemma":  "google/gemma-2-2b-it",          # stand-in for Gemma 4 E2B until released
    "phi4":   "microsoft/Phi-4-mini-instruct",
}

# Bake-off uses small slice for fast comparison; full SFT is in 02_train_tier1.
TRAIN_LIMIT = 100        # examples for QLoRA fine-tune
EVAL_LIMIT  = 20         # held-out examples for inference comparison
MAX_SEQ_LEN = 1024
EPOCHS      = 1
BATCH_SIZE  = 4
GRAD_ACCUM  = 4
LR          = 2e-4
LORA_R      = 32
LORA_ALPHA  = 64

OUT_DIR = "/content/yen_sei_bakeoff"
os.makedirs(OUT_DIR, exist_ok=True)
print(f"Candidates: {list(CANDIDATES.values())}")
print(f"Output dir: {OUT_DIR}")


## Upload data

The next cell looks for `train.jsonl` and `val.jsonl` in `/content/`. If missing, it opens an upload widget — drop both files from `tools/yen_sei/data/refined/`. (You can also mount Drive and copy them manually instead — see the cell.)


In [ ]:
# =====================================================================
# Cell 3 — Locate (or upload) train.jsonl and val.jsonl.
# =====================================================================
import json
from pathlib import Path

REQUIRED = ["train.jsonl", "val.jsonl"]
SEARCH_DIRS = [Path("/content"), Path.cwd(), Path("/content/drive/MyDrive/yen_sei")]

def find_data():
    found = {}
    for name in REQUIRED:
        for d in SEARCH_DIRS:
            p = d / name
            if p.exists():
                found[name] = p
                break
    return found

found = find_data()
missing = [n for n in REQUIRED if n not in found]

if missing:
    print(f"Missing: {missing}. Opening upload widget…")
    try:
        from google.colab import files  # type: ignore
        uploaded = files.upload()
        for name in uploaded:
            dest = Path("/content") / name
            print(f"  saved {name} -> {dest} ({len(uploaded[name])} bytes)")
        found = find_data()
    except ImportError:
        raise SystemExit(
            "Not in Colab. Place train.jsonl and val.jsonl next to this notebook "
            "or in /content/."
        )

assert all(n in found for n in REQUIRED), f"Still missing: {set(REQUIRED) - set(found)}"
TRAIN_PATH = found["train.jsonl"]
VAL_PATH = found["val.jsonl"]
print(f"\ntrain: {TRAIN_PATH}\nval:   {VAL_PATH}")

def load_jsonl(path, limit=None):
    rows = []
    with open(path, "r", encoding="utf-8-sig") as f:
        for line in f:
            rows.append(json.loads(line))
            if limit and len(rows) >= limit:
                break
    return rows

train_rows = load_jsonl(TRAIN_PATH, limit=TRAIN_LIMIT)
eval_rows = load_jsonl(VAL_PATH, limit=EVAL_LIMIT)
print(f"\nLoaded {len(train_rows)} train rows, {len(eval_rows)} eval rows.")
print(f"First example keys: {list(train_rows[0].keys())}")
print(f"First example messages roles: {[m['role'] for m in train_rows[0]['messages']]}")


In [ ]:
# =====================================================================
# Cell 4 - Helpers: train one candidate, run inference, score (Layer A+B).
# =====================================================================
import gc, json, re, time, torch

JSON_BLOCK_RE = re.compile(r"\{.*\}", re.DOTALL)
SGF_COORD_RE  = re.compile(r"\b[a-s]{2}\b")
TAGS_RE       = re.compile(r"^Tags: (.+)$",         re.MULTILINE)
BLACK_RE      = re.compile(r"^Black stones: (.+)$", re.MULTILINE)
WHITE_RE      = re.compile(r"^White stones: (.+)$", re.MULTILINE)
HINT_COORD_RE = re.compile(r"\{!([a-s]{2})\}")
TECHNIQUE_WORDS = {
    "ladder","snapback","net","throw-in","throw","tesuji","atari","ko","seki","eye",
    "sente","gote","shape","shicho","geta","wedge","hane","cut","connect","capture",
    "kill","live","life","death","endgame","joseki","fuseki","yose",
}

def _vram_peak_gb():
    return torch.cuda.max_memory_allocated() / (1024 ** 3) if torch.cuda.is_available() else 0.0

def _reset_vram():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()

def train_one(model_id, train_rows, out_subdir):
    from unsloth import FastLanguageModel  # type: ignore
    from datasets import Dataset
    from trl import SFTTrainer, SFTConfig
    _reset_vram()
    print(f"\n=== {model_id} ===")
    t0 = time.time()
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=model_id, max_seq_length=MAX_SEQ_LEN, load_in_4bit=True, dtype=None,
    )
    model = FastLanguageModel.get_peft_model(
        model, r=LORA_R, lora_alpha=LORA_ALPHA, target_modules="all-linear",
        lora_dropout=0.0, bias="none", use_gradient_checkpointing="unsloth",
    )
    def _format(example):
        return {"text": tokenizer.apply_chat_template(
            example["messages"], tokenize=False, add_generation_prompt=False)}
    ds = Dataset.from_list(train_rows).map(_format, remove_columns=["messages"])
    out_dir = os.path.join(OUT_DIR, out_subdir)
    trainer = SFTTrainer(
        model=model, tokenizer=tokenizer, train_dataset=ds,
        dataset_text_field="text", max_seq_length=MAX_SEQ_LEN,
        args=SFTConfig(
            output_dir=out_dir, num_train_epochs=EPOCHS,
            per_device_train_batch_size=BATCH_SIZE, gradient_accumulation_steps=GRAD_ACCUM,
            learning_rate=LR, lr_scheduler_type="cosine", warmup_ratio=0.1,
            weight_decay=0.01, logging_steps=10,
            bf16=torch.cuda.is_bf16_supported(), fp16=not torch.cuda.is_bf16_supported(),
            optim="adamw_8bit", report_to="none", save_strategy="no",
        ),
    )
    trainer.train()
    train_secs = time.time() - t0
    peak = _vram_peak_gb()
    print(f"  train: {train_secs:.0f}s, peak VRAM: {peak:.1f} GB")
    return model, tokenizer, train_secs, peak

def generate_answer(model, tokenizer, messages, max_new_tokens=384):
    from unsloth import FastLanguageModel  # type: ignore
    FastLanguageModel.for_inference(model)
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    t0 = time.time()
    with torch.no_grad():
        outputs = model.generate(
            **inputs, max_new_tokens=max_new_tokens, do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )
    text = tokenizer.decode(outputs[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    return text, outputs.shape[1] - inputs["input_ids"].shape[1], time.time() - t0

def _parse_user(text):
    tags  = [t.strip() for t in (TAGS_RE.search(text).group(1).split(",")  if TAGS_RE.search(text)  else [])]
    black = [c.strip() for c in (BLACK_RE.search(text).group(1).split(",") if BLACK_RE.search(text) else [])]
    white = [c.strip() for c in (WHITE_RE.search(text).group(1).split(",") if WHITE_RE.search(text) else [])]
    return tags, black, white

def score_output(generated, reference, user_text):
    """Returns merged Layer-A + Layer-B score dict."""
    s = {"parsed_ok": False, "has_correct": False, "has_wrong": False, "n_hints": 0,
         "n_chars_correct": 0, "n_chars_wrong": 0,
         "mentions_correct_move": False, "mentions_tag_technique": False,
         "no_off_board_coords": True, "looks_english": False}
    flat_text = generated or ""
    blob = JSON_BLOCK_RE.search(flat_text)
    if blob:
        try:
            obj = json.loads(blob.group(0))
            s["parsed_ok"] = True
            tc = obj.get("teaching_comments") or {}
            cc = (tc.get("correct_comment") or "").strip() if isinstance(tc, dict) else ""
            wc = tc.get("wrong_comments") if isinstance(tc, dict) else None
            if isinstance(wc, dict):    wc_text = " ".join(str(v) for v in wc.values())
            elif isinstance(wc, list):  wc_text = " ".join(str(v) for v in wc)
            else:                       wc_text = ""
            s["has_correct"]     = bool(cc)
            s["has_wrong"]       = bool(wc_text.strip())
            s["n_chars_correct"] = len(cc)
            s["n_chars_wrong"]   = len(wc_text)
            hints = obj.get("hints") or []
            s["n_hints"] = len(hints) if isinstance(hints, list) else 0
            parts = []
            def walk(x):
                if isinstance(x, str): parts.append(x)
                elif isinstance(x, dict):
                    for v in x.values(): walk(v)
                elif isinstance(x, list):
                    for v in x: walk(v)
            walk(obj)
            flat_text = " ".join(parts)
        except json.JSONDecodeError:
            pass

    tags, black, white = _parse_user(user_text)
    correct_move = ""
    if reference:
        m = HINT_COORD_RE.search(reference)
        if m: correct_move = m.group(1)
    text_lower = flat_text.lower()
    if correct_move and correct_move in text_lower:
        s["mentions_correct_move"] = True
    tag_tokens = set()
    for t in tags:
        for piece in re.split(r"[-_,\s]+", t.lower()):
            if piece: tag_tokens.add(piece)
    interesting = tag_tokens & TECHNIQUE_WORDS
    if interesting:
        s["mentions_tag_technique"] = any(w in text_lower for w in interesting)
    else:
        s["mentions_tag_technique"] = True  # vacuous pass when no recognizable techniques
    coord_set = {c.lower() for c in black + white}
    if correct_move: coord_set.add(correct_move)
    found = SGF_COORD_RE.findall(text_lower)
    unknown = [c for c in found if c not in coord_set]
    s["no_off_board_coords"] = len(unknown) <= max(1, len(found) // 3)
    letters = sum(1 for ch in flat_text if "a" <= ch.lower() <= "z")
    s["looks_english"] = (letters / max(len(flat_text), 1)) >= 0.6 if flat_text else False
    return s

def evaluate_model(model, tokenizer, eval_rows):
    rows = []
    for ex in eval_rows:
        msgs        = ex["messages"]
        prompt_msgs = [m for m in msgs if m["role"] != "assistant"]
        reference   = next((m["content"] for m in msgs if m["role"] == "assistant"), None)
        user_text   = next((m["content"] for m in msgs if m["role"] == "user"), "")
        text, n_tok, secs = generate_answer(model, tokenizer, prompt_msgs)
        sc = score_output(text, reference, user_text)
        sc.update({"n_tokens": n_tok, "gen_seconds": secs,
                   "generated_preview": text[:600], "reference_preview": (reference or "")[:600],
                   "user_preview": user_text[:400]})
        rows.append(sc)
    return rows

def aggregate(eval_rows):
    n = len(eval_rows) or 1
    pct = lambda key: round(100.0 * sum(1 for r in eval_rows if r.get(key)) / n, 1)
    useful = sum(
        1 for r in eval_rows
        if r["parsed_ok"] and r["has_correct"]
           and (r["mentions_correct_move"] or r["mentions_tag_technique"])
           and r["looks_english"]
    )
    return {
        "useful_answer_pct":          round(100.0 * useful / n, 1),
        "json_compliance_pct":        pct("parsed_ok"),
        "has_correct_pct":            pct("has_correct"),
        "has_wrong_pct":              pct("has_wrong"),
        "mentions_correct_move_pct":  pct("mentions_correct_move"),
        "mentions_tag_technique_pct": pct("mentions_tag_technique"),
        "no_off_board_coords_pct":    pct("no_off_board_coords"),
        "looks_english_pct":          pct("looks_english"),
        "avg_hints":                  round(sum(r["n_hints"] for r in eval_rows) / n, 2),
        "avg_chars_correct":          round(sum(r["n_chars_correct"] for r in eval_rows) / n, 1),
        "avg_tok_per_sec":            round(sum(r["n_tokens"] / max(r["gen_seconds"], 1e-6) for r in eval_rows) / n, 1),
    }

def free_model(model):
    del model
    _reset_vram()

print("Helpers ready (Layer A+B scoring; Layer C is manual review of dump).")


## Train + evaluate Candidate A (Gemma)


In [ ]:
# =====================================================================
# Cell 5 — Bake-off Candidate A.
# =====================================================================
results = {}

m, tok, train_secs, peak = train_one(CANDIDATES["gemma"], train_rows, "gemma")
eval_rows_a = evaluate_model(m, tok, eval_rows)
agg_a = aggregate(eval_rows_a)
agg_a["model"] = CANDIDATES["gemma"]
agg_a["train_seconds"] = train_secs
agg_a["peak_vram_gb"] = peak
results["gemma"] = agg_a
print(json.dumps(agg_a, indent=2))

free_model(m)


In [ ]:
# =====================================================================
# Cell 6 — Bake-off Candidate B.
# =====================================================================
m, tok, train_secs, peak = train_one(CANDIDATES["phi4"], train_rows, "phi4")
eval_rows_b = evaluate_model(m, tok, eval_rows)
agg_b = aggregate(eval_rows_b)
agg_b["model"] = CANDIDATES["phi4"]
agg_b["train_seconds"] = train_secs
agg_b["peak_vram_gb"] = peak
results["phi4"] = agg_b
print(json.dumps(agg_b, indent=2))

free_model(m)


## Verdict


In [ ]:
# =====================================================================
# Cell 7 — Compare and pick winner.
# =====================================================================

def fmt(v, w=10, d=1):
    if isinstance(v, (int, float)):
        return f"{v:>{w}.{d}f}"
    return f"{v:>{w}}"

a, b = results["gemma"], results["phi4"]
rows = [
    ("model",                 a["model"],                b["model"]),
    ("params (B)",            "2.6",                     "3.8"),
    ("train sec (1 epoch)",   f"{a['train_seconds']:.0f}", f"{b['train_seconds']:.0f}"),
    ("peak VRAM (GB)",        f"{a['peak_vram_gb']:.1f}",  f"{b['peak_vram_gb']:.1f}"),
    ("JSON compliance %",     f"{a['json_compliance_pct']:.1f}", f"{b['json_compliance_pct']:.1f}"),
    ("has correct_comment %", f"{a['has_correct_pct']:.1f}",     f"{b['has_correct_pct']:.1f}"),
    ("has wrong_comments %",  f"{a['has_wrong_pct']:.1f}",       f"{b['has_wrong_pct']:.1f}"),
    ("has hints %",           f"{a['has_hints_pct']:.1f}",       f"{b['has_hints_pct']:.1f}"),
    ("avg hints",             f"{a['avg_hints']:.2f}",           f"{b['avg_hints']:.2f}"),
    ("avg output tokens",     f"{a['avg_tokens']:.0f}",          f"{b['avg_tokens']:.0f}"),
    ("inference tok/s",       f"{a['avg_tok_per_sec']:.1f}",     f"{b['avg_tok_per_sec']:.1f}"),
]
print(f"{'metric':<25} {'gemma':>30} {'phi4':>30}")
print("-" * 90)
for label, av, bv in rows:
    print(f"{label:<25} {str(av):>30} {str(bv):>30}")

# Decision: highest JSON compliance wins; tie-break = smaller model.
if a["json_compliance_pct"] > b["json_compliance_pct"]:
    winner_key, winner = "gemma", a
elif b["json_compliance_pct"] > a["json_compliance_pct"]:
    winner_key, winner = "phi4", b
else:
    # Tie -> prefer smaller model (gemma)
    winner_key, winner = "gemma", a

verdict_path = Path(OUT_DIR) / "winner.json"
verdict_path.write_text(json.dumps({
    "winner": winner_key,
    "model_id": winner["model"],
    "metrics": winner,
    "rationale": "Higher JSON-schema compliance on held-out eval (tie-break: smaller model).",
}, indent=2))
print(f"\nWINNER: {winner_key}  ({winner['model']})")
print(f"Saved verdict -> {verdict_path}")
print(f"\nNext step: open 02_train_tier1.ipynb, set MODEL_NAME = '{winner['model']}', Run all.")


In [ ]:
# =====================================================================
# Cell 8 — Zip and download the verdict + per-example eval traces.
# =====================================================================
import shutil
from pathlib import Path

artifact = Path(OUT_DIR) / "bakeoff_artifacts"
artifact.mkdir(exist_ok=True)
(artifact / "winner.json").write_bytes((Path(OUT_DIR) / "winner.json").read_bytes())
(artifact / "results.json").write_text(json.dumps(results, indent=2))
(artifact / "eval_gemma.json").write_text(json.dumps(eval_rows_a, indent=2))
(artifact / "eval_phi4.json").write_text(json.dumps(eval_rows_b, indent=2))

zip_path = shutil.make_archive(str(Path(OUT_DIR) / "bakeoff_results"), "zip", artifact)
print(f"Zipped -> {zip_path}")

try:
    from google.colab import files  # type: ignore
    files.download(zip_path)
except ImportError:
    print(f"Not in Colab. Files saved at: {artifact}")
